## Purpose of this notebook
(Author: Siqi Xiang)

### Artifact Review & Business Sanity Check

This notebook reviews the final artifacts produced by the batch churn scoring pipeline.

Goals:
- Inspect the Top-K targeting list delivered to operations
- Validate ranking logic and incentive recommendations
- Understand how capacity constraints affect expected value
- Translate model outputs into business decisions

In [8]:
import pandas as pd

scores = pd.read_parquet(
    "s3://online-retail-churn-siqi-dev/scores/online_retail/dt=2026-01-12/ref=2011-10-10/scores.parquet",
    storage_options={"profile": "siqi-dev"}
)

scores.shape, scores.columns.tolist()

((3013, 14),
 ['CustomerID',
  'reference_date',
  'churn_60d',
  'recency_days',
  'frequency_180d',
  'monetary_180d',
  'aov_180d',
  'risk_score',
  'value_segment',
  'recommended_action',
  'recommended_coupon_eur',
  'ev',
  'dt',
  'ref'])

In [9]:
scores["value_segment"].value_counts(dropna=False), scores["recommended_coupon_eur"].value_counts(dropna=False)

(value_segment
 high    1005
 low     1005
 mid     1003
 Name: count, dtype: Int64,
 recommended_coupon_eur
 5     2008
 10    1005
 Name: count, dtype: int64)

In [10]:
K = 500
ranked = (
    scores
    .sort_values("ev", ascending=False)
    .head(K)
    .reset_index(drop=True)
)

ranked.head(10)[["CustomerID","risk_score","value_segment","recommended_coupon_eur","recommended_action","ev"]]

,CustomerID,risk_score,value_segment,recommended_coupon_eur,recommended_action,ev
0,15098,0.491454,high,10,send_coupon,5875.133492
1,15749,0.688807,high,10,send_coupon,4440.222549
2,17450,0.086998,high,10,send_coupon,3305.496893
3,18102,0.064761,high,10,send_coupon,3057.097861
4,14088,0.353510,high,10,send_coupon,1895.026791
5,12590,0.578327,high,10,send_coupon,1701.431104
6,14646,0.039661,high,10,send_coupon,1445.602451
7,12415,0.058704,high,10,send_coupon,1405.356509
8,13081,0.473034,high,10,send_coupon,1276.054210
9,14298,0.111724,high,10,send_coupon,991.863035


In [4]:
import pandas as pd

scores = pd.read_parquet(
    "s3://online-retail-churn-siqi-dev/scores/online_retail/dt=2026-01-12/ref=2011-10-10/scores.parquet",
    storage_options={"profile": "siqi-dev"}
)

scores.head()

,CustomerID,reference_date,churn_60d,recency_days,frequency_180d,monetary_180d,aov_180d,risk_score,value_segment,recommended_action,recommended_coupon_eur,ev,dt,ref
0,13120,2011-10-10,1,178,1,30.60,30.60,0.938807,low,no_action,5,-3.563626,2026-01-12,2011-10-10
1,15139,2011-10-10,1,179,1,56.86,56.86,0.938060,low,no_action,5,-2.333095,2026-01-12,2011-10-10
2,17746,2011-10-10,1,180,1,110.25,110.25,0.930178,low,send_coupon,5,0.127604,2026-01-12,2011-10-10
3,15179,2011-10-10,0,173,1,67.50,67.50,0.929265,low,no_action,5,-1.863731,2026-01-12,2011-10-10
4,15935,2011-10-10,1,179,1,108.04,108.04,0.929265,low,send_coupon,5,0.019889,2026-01-12,2011-10-10


ranked["recommended_coupon_eur"].value_counts()

In [13]:
ranked["recommended_action"].value_counts()

recommended_action
send_coupon    500
Name: count, dtype: int64

In [15]:
ranked["ev"].describe()

count     500.000000
mean      276.530369
std       408.137189
min       129.904842
25%       156.899503
50%       189.458425
75%       256.063962
max      5875.133492
Name: ev, dtype: float64

In [16]:
ranked["ev"].sum()

np.float64(138265.18436125125)

### Observation

Under the current policy assumptions, the EV-optimal Top-500 list consists entirely of high-value customers receiving €10 incentives.
This is because the combination of high customer value and strong assumed uplift (30%) dominates lower-cost alternatives in EV ranking.

While earlier strategy-level analysis shows that segmentation adds value mainly at larger capacity, execution-level ranking on real data indicates that, for this run and capacity, focusing exclusively on high-value customers maximizes expected return.